In [1]:
# ============================================================
# MINI PROJECT KRIPTOGRAFI
# SISTEM VERIFIKASI KEASLIAN DOKUMEN DIGITAL
# MENGGUNAKAN ALGORITMA HASH SHA-256
# Google Colab Version - LENGKAP & DIPERBAIKI
# ============================================================


# ============================================================
# CELL 1 - IMPORT LIBRARY
# ============================================================

import hashlib
import json
import os
import time
from datetime import datetime
from google.colab import files

print("=" * 51)
print("   SHA-256 DOCUMENT VERIFIER SYSTEM")
print("=" * 51)
print("Semua library berhasil diimport.")


# ============================================================
# CELL 2 - FUNGSI HASH SHA-256 UNTUK TEKS
# ============================================================

def sha256_text(text):
    """Menghitung hash SHA-256 dari sebuah string teks."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

# Contoh penggunaan
sample_text = "Dokumen Digital Aman"
hash_result = sha256_text(sample_text)

print("\n" + "=" * 51)
print(" CONTOH HASH SHA-256 TEKS")
print("=" * 51)
print("TEXT    :", sample_text)
print("SHA-256 :", hash_result)


# ============================================================
# CELL 3 - FUNGSI HASH SHA-256 UNTUK FILE
# ============================================================

def sha256_file(filename):
    """
    Menghitung hash SHA-256 dari sebuah file.
    Membaca file per blok 4096 byte agar efisien untuk file besar.
    """
    sha256 = hashlib.sha256()
    with open(filename, 'rb') as f:
        while True:
            data = f.read(4096)
            if not data:
                break
            sha256.update(data)
    return sha256.hexdigest()

print("\n[OK] Fungsi sha256_file() berhasil didefinisikan.")


# ============================================================
# CELL 4 - UPLOAD DAN HASH FILE
# ============================================================

print("\n" + "=" * 51)
print(" CELL 4: UPLOAD FILE & HITUNG HASH")
print("=" * 51)
print("Silakan upload file untuk dihitung hash SHA-256-nya.")

uploaded = files.upload()

if not uploaded:
    print("[PERINGATAN] Tidak ada file yang diupload.")
else:
    for filename in uploaded.keys():
        start_time = time.time()
        hash_value = sha256_file(filename)
        end_time = time.time()
        file_size = os.path.getsize(filename)

        print("\n" + "-" * 40)
        print("NAMA FILE :", filename)
        print("UKURAN    :", file_size, "bytes")
        print("SHA-256   :", hash_value)
        print("WAKTU     :", round(end_time - start_time, 6), "detik")
        print("-" * 40)
        print("[!] Salin hash di atas untuk keperluan verifikasi.")


# ============================================================
# CELL 5 - VERIFIKASI KEASLIAN FILE
# ============================================================

def verify_file(filename, original_hash):
    """
    Memverifikasi apakah file cocok dengan hash referensi.
    Mengembalikan (True, hash_saat_ini) jika cocok, (False, hash_saat_ini) jika tidak.
    """
    current_hash = sha256_file(filename)
    if current_hash.lower() == original_hash.lower():
        return True, current_hash
    else:
        return False, current_hash

print("\n" + "=" * 51)
print(" CELL 5: VERIFIKASI KEASLIAN FILE")
print("=" * 51)
print("Upload file yang ingin diverifikasi.")

uploaded_verify = files.upload()

if not uploaded_verify:
    print("[PERINGATAN] Tidak ada file yang diupload untuk verifikasi.")
else:
    for filename in uploaded_verify.keys():
        reference_hash = input(f"\nMasukkan hash SHA-256 referensi untuk '{filename}':\n> ").strip()

        if not reference_hash:
            print("[ERROR] Hash referensi tidak boleh kosong!")
        else:
            status, generated_hash = verify_file(filename, reference_hash)

            print("\nHASH FILE SAAT INI:")
            print(generated_hash)
            print("\nHASH REFERENSI    :")
            print(reference_hash)

            print("\n" + "=" * 40)
            if status:
                print("STATUS : ✅ VALID")
                print("Dokumen ASLI dan tidak berubah.")
            else:
                print("STATUS : ❌ INVALID")
                print("Dokumen telah DIMODIFIKASI atau berbeda!")
            print("=" * 40)


# ============================================================
# CELL 6 - DEMONSTRASI AVALANCHE EFFECT
# ============================================================

def hitung_perbedaan_bit(hash1, hash2):
    """
    Menghitung jumlah bit yang berbeda antara dua hash hex.
    Mengkonversi hex ke binary terlebih dahulu.
    """
    bin1 = bin(int(hash1, 16))[2:].zfill(256)
    bin2 = bin(int(hash2, 16))[2:].zfill(256)
    return sum(a != b for a, b in zip(bin1, bin2))

print("\n" + "=" * 51)
print(" CELL 6: DEMONSTRASI AVALANCHE EFFECT")
print("=" * 51)
print("Masukkan dua teks yang mirip untuk melihat perbedaan hash.\n")

text1 = input("Masukkan teks pertama:\n> ")
text2 = input("\nMasukkan teks kedua:\n> ")

hash1 = sha256_text(text1)
hash2 = sha256_text(text2)

print("\n--- Teks & Hash ---")
print(f"TEKS 1  : {text1}")
print(f"HASH 1  : {hash1}")
print(f"\nTEKS 2  : {text2}")
print(f"HASH 2  : {hash2}")

# Perbedaan karakter hex
char_diff = sum(a != b for a, b in zip(hash1, hash2))
char_pct  = (char_diff / len(hash1)) * 100

# Perbedaan bit (lebih akurat untuk avalanche)
bit_diff = hitung_perbedaan_bit(hash1, hash2)
bit_pct  = (bit_diff / 256) * 100

print("\n--- Analisis Perbedaan ---")
print(f"Karakter hex berbeda : {char_diff} / {len(hash1)} ({round(char_pct, 2)}%)")
print(f"Bit berbeda          : {bit_diff} / 256 ({round(bit_pct, 2)}%)")

print("\n" + "=" * 40)
if bit_pct >= 45:
    print("✅ Avalanche Effect TERBUKTI")
    print(f"   Sekitar {round(bit_pct,1)}% bit berubah.")
else:
    print("⚠️  Avalanche Effect kurang signifikan")
    print(f"   Hanya {round(bit_pct,1)}% bit yang berubah.")
print("=" * 40)


# ============================================================
# CELL 7 - SISTEM LOG RIWAYAT
# ============================================================

logs = []

def add_log(activity, filename, hash_value, status):
    """Menambahkan entri log ke dalam daftar logs."""
    log_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "activity": activity,
        "filename": filename,
        "hash": hash_value,
        "status": status
    }
    logs.append(log_data)
    return log_data

# Tambah log contoh otomatis
add_log("HASHING",      "contoh.pdf",  "abc123def456", "SUCCESS")
add_log("VERIFIKASI",   "laporan.docx","789xyz012uvw", "VALID")
add_log("MODIFIKASI",   "data.csv",    "111aaa222bbb", "INVALID")

print("\n" + "=" * 51)
print(" CELL 7: SISTEM LOG RIWAYAT")
print("=" * 51)
print(f"[OK] {len(logs)} log contoh berhasil ditambahkan.")

# Tambah log interaktif (opsional)
tambah = input("\nApakah ingin menambah log manual? (y/n): ").strip().lower()
if tambah == 'y':
    act  = input("Aktivitas (HASHING/VERIFIKASI/dll): ").strip().upper()
    fn   = input("Nama file: ").strip()
    hv   = input("Hash value: ").strip()
    st   = input("Status (SUCCESS/VALID/INVALID): ").strip().upper()
    new_log = add_log(act, fn, hv, st)
    print(f"[OK] Log ditambahkan: {new_log}")


# ============================================================
# CELL 8 - TAMPILKAN LOG
# ============================================================

print("\n" + "=" * 51)
print(" CELL 8: RIWAYAT AKTIVITAS")
print("=" * 51)

if not logs:
    print("Belum ada log.")
else:
    for i, log in enumerate(logs, 1):
        print(f"\n{'─'*38}")
        print(f"  Log Ke-{i}")
        print(f"{'─'*38}")
        print(f"  Waktu     : {log['timestamp']}")
        print(f"  Aktivitas : {log['activity']}")
        print(f"  File      : {log['filename']}")
        print(f"  Hash      : {log['hash']}")
        print(f"  Status    : {log['status']}")
    print(f"\nTotal log: {len(logs)}")


# ============================================================
# CELL 9 - EXPORT LOG KE JSON
# ============================================================

export_data = {
    "exported_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_logs": len(logs),
    "logs": logs
}

json_filename = "sha256_logs.json"
with open(json_filename, "w") as f:
    json.dump(export_data, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 51)
print(" CELL 9: EXPORT LOG KE JSON")
print("=" * 51)
print(f"[OK] File '{json_filename}' berhasil dibuat.")
print(f"     Total log diekspor: {len(logs)}")

files.download(json_filename)
print(f"[OK] File '{json_filename}' sedang diunduh.")


# ============================================================
# CELL 10 - PENGUJIAN INTEGRITAS HASH (DETERMINISME)
# ============================================================

print("\n" + "=" * 51)
print(" CELL 10: PENGUJIAN INTEGRITAS HASH")
print("=" * 51)

test_text = "Keamanan Dokumen Digital"
hash_a = sha256_text(test_text)
hash_b = sha256_text(test_text)

print(f"Teks Uji : '{test_text}'")
print(f"\nHASH 1   : {hash_a}")
print(f"HASH 2   : {hash_b}")

print("\n" + "-" * 40)
if hash_a == hash_b:
    print("HASIL : ✅ PASS - Hash konsisten (deterministik).")
else:
    print("HASIL : ❌ FAIL - Hash tidak konsisten!")
print("-" * 40)


# ============================================================
# CELL 11 - PENGUJIAN SENSITIVITAS (1 KARAKTER BERBEDA)
# ============================================================

print("\n" + "=" * 51)
print(" CELL 11: PENGUJIAN MODIFIKASI 1 KARAKTER")
print("=" * 51)

original = "Dokumen Asli"
modified = "Dokumen asli"   # Hanya huruf kapital 'A' → 'a'

original_hash = sha256_text(original)
modified_hash = sha256_text(modified)

print(f"TEXT ORIGINAL   : '{original}'")
print(f"HASH ORIGINAL   : {original_hash}")
print(f"\nTEXT MODIFIKASI : '{modified}'")
print(f"HASH MODIFIKASI : {modified_hash}")

bit_diff_test = hitung_perbedaan_bit(original_hash, modified_hash)
print(f"\nPerbedaan bit   : {bit_diff_test} / 256 ({round(bit_diff_test/256*100, 2)}%)")

print("\n" + "-" * 40)
if original_hash != modified_hash:
    print("STATUS : ✅ MODIFIKASI TERDETEKSI")
    print("SHA-256 berhasil mendeteksi perubahan 1 karakter.")
else:
    print("STATUS : ❌ TIDAK TERDETEKSI (seharusnya tidak terjadi)")
print("-" * 40)


# ============================================================
# CELL 12 - RINGKASAN & PENUTUP
# ============================================================

print("\n" + "=" * 51)
print("   SISTEM VERIFIKASI DOKUMEN SHA-256 SELESAI")
print("=" * 51)

print("""
Fitur yang berhasil diimplementasikan:

  1. ✅ Hash SHA-256 teks
  2. ✅ Hash SHA-256 file (streaming, efisien)
  3. ✅ Verifikasi keaslian dokumen
  4. ✅ Demonstrasi Avalanche Effect (bit & karakter)
  5. ✅ Sistem log riwayat
  6. ✅ Export log JSON
  7. ✅ Pengujian integritas hash (determinisme)
  8. ✅ Pengujian sensitivitas 1 karakter

Catatan Keamanan:
  - SHA-256 menghasilkan hash 256-bit (64 karakter hex)
  - Setiap perubahan sekecil apapun menghasilkan hash berbeda
  - Hash bersifat one-way (tidak bisa di-reverse)
  - Cocok untuk verifikasi integritas file & dokumen digital
""")
print("=" * 51)

   SHA-256 DOCUMENT VERIFIER SYSTEM
Semua library berhasil diimport.

 CONTOH HASH SHA-256 TEKS
TEXT    : Dokumen Digital Aman
SHA-256 : 810652e494ea651c992673cf11a4eb4af2daa5969efc5604bfd03ba7eac2e0ae

[OK] Fungsi sha256_file() berhasil didefinisikan.

 CELL 4: UPLOAD FILE & HITUNG HASH
Silakan upload file untuk dihitung hash SHA-256-nya.


Saving kernel rbf.pdf to kernel rbf.pdf

----------------------------------------
NAMA FILE : kernel rbf.pdf
UKURAN    : 439463 bytes
SHA-256   : 4a5d43d7a6320c298e58405a22dcb74e98ad2e942734c4f03fabdec52991567d
WAKTU     : 0.001579 detik
----------------------------------------
[!] Salin hash di atas untuk keperluan verifikasi.

 CELL 5: VERIFIKASI KEASLIAN FILE
Upload file yang ingin diverifikasi.


Saving kernel rbf.pdf to kernel rbf (1).pdf

Masukkan hash SHA-256 referensi untuk 'kernel rbf (1).pdf':
> 4a5d43d7a6320c298e58405a22dcb74e98ad2e942734c4f03fabdec52991567d

HASH FILE SAAT INI:
4a5d43d7a6320c298e58405a22dcb74e98ad2e942734c4f03fabdec52991567d

HASH REFERENSI    :
4a5d43d7a6320c298e58405a22dcb74e98ad2e942734c4f03fabdec52991567d

STATUS : ✅ VALID
Dokumen ASLI dan tidak berubah.

 CELL 6: DEMONSTRASI AVALANCHE EFFECT
Masukkan dua teks yang mirip untuk melihat perbedaan hash.

Masukkan teks pertama:
> open AI

Masukkan teks kedua:
> open AI

--- Teks & Hash ---
TEKS 1  : open AI
HASH 1  : 2b6fef6fc73f7a572006f0536f16e14f4e4135f3811b06803b40094026a1addc

TEKS 2  : open AI
HASH 2  : 2b6fef6fc73f7a572006f0536f16e14f4e4135f3811b06803b40094026a1addc

--- Analisis Perbedaan ---
Karakter hex berbeda : 0 / 64 (0.0%)
Bit berbeda          : 0 / 256 (0.0%)

⚠️  Avalanche Effect kurang signifikan
   Hanya 0.0% bit yang berubah.

 CELL 7: SISTEM LOG RIWAYAT
[OK] 3 log contoh berhasil dit

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] File 'sha256_logs.json' sedang diunduh.

 CELL 10: PENGUJIAN INTEGRITAS HASH
Teks Uji : 'Keamanan Dokumen Digital'

HASH 1   : 4188c3718585cdceff2b4a4585eca76b7ac2b22d1782ae485445bb2c0dd986b9
HASH 2   : 4188c3718585cdceff2b4a4585eca76b7ac2b22d1782ae485445bb2c0dd986b9

----------------------------------------
HASIL : ✅ PASS - Hash konsisten (deterministik).
----------------------------------------

 CELL 11: PENGUJIAN MODIFIKASI 1 KARAKTER
TEXT ORIGINAL   : 'Dokumen Asli'
HASH ORIGINAL   : 5193b3d36fce462cb44da5a546ff2fcfa032df9ebd3decf4d493d71eeda6f618

TEXT MODIFIKASI : 'Dokumen asli'
HASH MODIFIKASI : a34682d97156f607b52350ed5df0895852fd7180a4d8383380e0c6529c9c3a33

Perbedaan bit   : 126 / 256 (49.22%)

----------------------------------------
STATUS : ✅ MODIFIKASI TERDETEKSI
SHA-256 berhasil mendeteksi perubahan 1 karakter.
----------------------------------------

   SISTEM VERIFIKASI DOKUMEN SHA-256 SELESAI

Fitur yang berhasil diimplementasikan:

  1. ✅ Hash SHA-256 teks
  2